# 🌌 Multiverse Synthetic Validation (Validación por Multiversos Paralelos)
**Objetivo:** Evaluar la solidez matemática y el edge predictivo del Motor Quant **Kinetopus** mediante el enfoque de "Múltiples Universos Paralelos" con datos sintéticos (**Camino 2**).

### 🧠 ¿Por qué el Enfoque Multiverso?
1. **Estabilidad Física y Límite Caótico:** En lugar de proyectar un único horizonte temporal extremadamente largo (ej. 50,000 velas) que magnifica el ruido y las inestabilidades numéricas de las ODEs, simulamos **$N$ universos paralelos independientes** de longitud estándar (ej. 2,000 velas) gobernados por el mismo atractor físico madre.
2. **Aislamiento en Memoria (RAM Sovereignty):** Toda la simulación ocurre **estrictamente en memoria RAM**. Las series de tiempo se generan, se validan mediante el `WalkForwardEvaluator` y sus variables se liberan inmediatamente (`gc.collect()`), garantizando un crecimiento de memoria $O(1)$ óptimo para laptops de 16GB RAM.
3. **Compatibilidad Estructural (Plug-and-Play):** Los resultados del Walk-Forward se exportan a una base de datos con columnas idénticas a las reales. Además, interceptamos las consultas direccionales guardando los precios de los universos en una caché local `.synthetic_prices.pkl`. Esto permite que el notebook [3_Analytics_and_Discovery.ipynb](file:///c:/Users/ussaa/Documents/KineTopus%20Engine/notebooks_val/3_Analytics_and_Discovery.ipynb) analice e interactúe con los multiversos **sin modificar una sola línea de su código**.

In [21]:
# Activar autoreload para recargar en caliente los módulos del core
%load_ext autoreload
%autoreload 2

import sys
import os
import gc
import pickle
import time
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Resolución dinámica de la raíz del proyecto para evitar ModuleNotFoundError
current_dir = os.path.abspath(os.getcwd())
while current_dir != os.path.dirname(current_dir):
    if 'src' in os.listdir(current_dir):
        if current_dir not in sys.path:
            sys.path.append(current_dir)
        break
    current_dir = os.path.dirname(current_dir)

# Kinetopus Core
from src.ui.market_loader import MarketLoader
from src.quant_engine.blender import ContinuousBlender
from src.quant_engine.physics import PhysicsDiscoverer
from src.quant_engine.nervous import RegimeShiftDetector
from src.quant_engine.evaluator import WalkForwardEvaluator

warnings.filterwarnings('ignore')
print("🤖 Kinetopus Core e interfaces vectorizadas cargadas con éxito!")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
🤖 Kinetopus Core e interfaces vectorizadas cargadas con éxito!


## ⚙️ 1. Configuración de Parámetros (El Panel de Control)
Aquí defines el tamaño de los universos y la cantidad de mundos alternativos a crear.

In [22]:
# ==============================================================================
# PANEL DE CONTROL DE SIMULACIÓN MULTIVERSO
# ==============================================================================
BASE_TICKER = 'BTC-USD'             # Activo de referencia (donador físico)
NUM_UNIVERSES = 2                  # Número de multiversos paralelos independientes
CANDLES_PER_UNIVERSE = 1300         # Tamaño muestral de cada universo (velas)
MC_TRAJECTORIES = 1000              # Resolucion de Monte Carlo para la proyección

# Parámetros de Walk-Forward a ciegas (Mismos del Notebook 1)
INITIAL_WINDOW = 360                # Ventana de entrenamiento inicial (velas)
STRIDE = 20                         # Salto iterativo (velas)
HORIZON = 150                       # Horizonte a ciegas del futuro
BLOCKS = 15                         # Bloques del horizonte (1 bloque = 10 velas)

# Rutas de Datos y Caché
DB_SYNTHETIC_PATH = "macro_backtest_synthetic_db.csv"
CACHE_PATH = ".synthetic_prices.pkl"

print(f"🌍 Configurado: {NUM_UNIVERSES} Multiversos de {CANDLES_PER_UNIVERSE} velas cada uno.")
print(f"🔍 Parámetros Walk-Forward: Ventana={INITIAL_WINDOW}, Salto={STRIDE}, Horizonte={HORIZON} ({BLOCKS} bloques).")

🌍 Configurado: 2 Multiversos de 1300 velas cada uno.
🔍 Parámetros Walk-Forward: Ventana=360, Salto=20, Horizonte=150 (15 bloques).


## 🧬 2. Extracción del Genoma Físico de Referencia
Descargamos las velas reales del activo base con `MarketLoader` y extraemos su dinámica continua y ecuaciones gobernantes utilizando SINDy bajo el régimen activo de CUSUM.

In [23]:
print(f"📥 Descargando histórico real para {BASE_TICKER}...")
df_real = MarketLoader.load_ticker_data(BASE_TICKER, period='1y', interval='1d')
if isinstance(df_real.columns, pd.MultiIndex):
    df_real.columns = df_real.columns.droplevel(1)

# Sanitización de log returns y volumen Z-Score
log_returns, vol, raw_price, dt_val = MarketLoader.prepare_quant_input(df_real, disable_norm=False, disable_returns=False)
mu_v = np.mean(vol)
sigma_v = np.std(vol) if np.std(vol) > 1e-8 else 1.0
vol_z = (vol - mu_v) / sigma_v

# Suavizado topológico continuo (Capa 2: Splines de Memoria Líquida)
t_hist = np.arange(len(df_real), dtype=np.float64)
blender = ContinuousBlender(tolerance=0.0025)
blender.fit(t_hist, log_returns, dominant_periods=np.array([]), feature_idx=0)
blender.fit(t_hist, vol_z, dominant_periods=np.array([]), feature_idx=1)

smooth_r, r_dot, _ = blender.compute_continuous(0, t_hist)
v_smooth, v_dot, _ = blender.compute_continuous(1, t_hist)

# Detección de Quiebres con CUSUM de la app Streamlit
detector = RegimeShiftDetector(threshold=5.0, drift=1.0)
cusum_report = detector.detect(log_returns, smooth_r)
shift_idx = cusum_report['shift_indices']

# Determinar ventana del régimen activo
boundaries = [0] + shift_idx + [len(t_hist)]
start_idx = 0
for i_b in range(len(boundaries) - 1):
    s_idx = boundaries[i_b]
    e_idx = boundaries[i_b+1]
    if e_idx - s_idx >= 15:
        start_idx = s_idx

t_slice = t_hist[start_idx:]
x_slice = np.column_stack((smooth_r, v_smooth))[start_idx:]
x_dot_slice = np.column_stack((r_dot, v_dot))[start_idx:]

sigma_res_r = float(np.std(log_returns[start_idx:] - smooth_r[start_idx:]))
sigma_res_v = float(np.std(vol_z[start_idx:] - v_smooth[start_idx:]))

# Ajustar descubridor SINDy para el régimen activo
discoverer = PhysicsDiscoverer(poly_degree=1)
report = discoverer.extract_equations(
    t=t_slice, x=x_slice, x_dot=x_dot_slice, dt=dt_val, 
    horizon_steps=CANDLES_PER_UNIVERSE,
    sigma_res_r=sigma_res_r, sigma_res_v=sigma_res_v,
    last_price=raw_price[-1], disable_norm=False, disable_returns=False,
    mc_paths=MC_TRAJECTORIES
)

print("\n" + "="*80)
print(" 🧬 ATRACTOR MATEMÁTICO BASE DESCUBIERTO (SINDY):")
print("="*80)
print(f"► dr/dt = {report['equations'][0]}")
print(f"► dV/dt = {report['equations'][1]}")
print(f"► R2 Score del Ajuste: {report['score']:.4f}")
print("="*80)

📥 Descargando histórico real para BTC-USD...

 🧬 ATRACTOR MATEMÁTICO BASE DESCUBIERTO (SINDY):
► dr/dt =  0.00147 1 +  0.00719 r + -0.00146 V
► dV/dt =  0.11206 1 +  0.32793 r + -0.03715 V
► R2 Score del Ajuste: 0.1312


## 🌌 3. El Bucle del Multiverso (Generación y Validación in-RAM)
Iteramos sobre los universos. Generamos la serie de precios con Euler-Maruyama alrededor de la ecuación descubierta, inyectamos volumen estocástico y variabilidad OHLC intradía, corremos la validación Walk-Forward a ciegas, registramos en caché in-memory y liberamos memoria RAM inmediatamente.

In [24]:
synthetic_universes_cache = {}
all_results = []

# Extraer el atractor principal proyectado (p50)
p_price = report['prediction']['price_percentiles']
if len(p_price) == 5:
    p50_path = np.array(p_price[2])  # Mediana de Monte Carlo
else:
    print("⚠️ Física inestable detectada, cayendo a extrapolación determinística desnuda.")
    p50_path = np.array(report['prediction']['det_price_path'])

if len(p50_path) < CANDLES_PER_UNIVERSE:
    raise ValueError(f"SINDy generó sólo {len(p50_path)} velas. Reduce CANDLES_PER_UNIVERSE o aumenta la estabilidad del atractor.")

# Media y std del volumen real histórico para simulación dinámica
mean_v = df_real['Volume'].mean() if 'Volume' in df_real.columns else 1000000.0
std_v = df_real['Volume'].std() if 'Volume' in df_real.columns else 150000.0

# Generación temporal futura consecutiva
if isinstance(df_real.index, pd.DatetimeIndex):
    start_date = df_real.index[-1] + pd.Timedelta(days=1)
    fechas = pd.date_range(start=start_date, periods=CANDLES_PER_UNIVERSE, freq='D')
else:
    fechas = np.arange(CANDLES_PER_UNIVERSE) + len(df_real)

print(f"🚀 Iniciando bucle de validación Walk-Forward en el Multiverso...")
start_time = time.time()

for i in range(1, NUM_UNIVERSES + 1):
    universe_ticker = f"{BASE_TICKER}_Synthetic_U{i}"
    print(f"\n🌌 [Universo {i}/{NUM_UNIVERSES}] Generando {universe_ticker}...")
    
    # 1. Simulación Estocástica del Precio (Caminos Paralelos)
    # Añadimos un shock browniano sutil exclusivo a cada universo para diversificar las trayectorias
    shock_vol = 0.0025
    random_drift = np.exp(np.cumsum(np.random.normal(0, shock_vol, size=CANDLES_PER_UNIVERSE)))
    close_sintetico = p50_path[:CANDLES_PER_UNIVERSE] * random_drift
    
    # 2. Modelado OHLC Intradía Coherente
    volatilidad_velas = 0.008
    open_sintetico = close_sintetico * (1.0 + np.random.normal(0, volatilidad_velas * 0.3, size=CANDLES_PER_UNIVERSE))
    high_sintetico = np.maximum(open_sintetico, close_sintetico) * (1.0 + np.abs(np.random.normal(0, volatilidad_velas * 0.5, size=CANDLES_PER_UNIVERSE)))
    low_sintetico = np.minimum(open_sintetico, close_sintetico) * (1.0 - np.abs(np.random.normal(0, volatilidad_velas * 0.5, size=CANDLES_PER_UNIVERSE)))
    
    # 3. Modelado de Volumen gaussiano robusto (Evita volumen plano)
    volume_sintetico = np.maximum(100.0, np.random.normal(mean_v, std_v * 0.20, size=CANDLES_PER_UNIVERSE))
    
    # 4. Consolidar el DataFrame de este Universo
    df_sintetico = pd.DataFrame({
        'Open': open_sintetico,
        'High': high_sintetico,
        'Low': low_sintetico,
        'Close': close_sintetico,
        'Volume': volume_sintetico
    }, index=fechas)
    df_sintetico.index.name = 'Date'
    
    # Guardar en la caché en memoria para la validación del Notebook 3
    synthetic_universes_cache[universe_ticker] = df_sintetico.copy()
    
    # 5. Ejecutar Walk-Forward a ciegas sobre el universo paralelo
    print(f"   🔎 Corriendo Evaluador Walk-Forward (Expanding Window)...")
    evaluador = WalkForwardEvaluator(df_sintetico, disable_norm=False, disable_returns=False)
    
    try:
        df_resultados = evaluador.run(
            initial_window=INITIAL_WINDOW, 
            stride=STRIDE, 
            horizon=HORIZON, 
            blocks=BLOCKS
        )
        
        if not df_resultados.empty:
            # Inyección de metadatos estructurales para compatibilidad analítica
            df_resultados.insert(0, 'Context_Window', 0)
            df_resultados.insert(0, 'Total_Velas_Disponible', CANDLES_PER_UNIVERSE)
            df_resultados.insert(0, 'Intervalo_Velas', 'Simulated')
            df_resultados.insert(0, 'Periodo_Historia', 'Synthetic')
            df_resultados.insert(0, 'Ticker', universe_ticker)
            
            all_results.append(df_resultados)
            print(f"   ✅ Éxito. Generadas {len(df_resultados)} iteraciones Walk-Forward.")
        else:
            print("   ⚠️ Advertencia: El evaluador no produjo ninguna iteración válida.")
            
    except Exception as e:
        print(f"   ❌ Error matemático/sistémico en Universo {i}: {e}")
        
    # 6. Garbage Collection (Crecimiento de RAM O(1) inquebrantable)
    del df_sintetico
    gc.collect()

print(f"\n🎉 Experimento Multiverso completado en {time.time() - start_time:.2f} segundos!")

🚀 Iniciando bucle de validación Walk-Forward en el Multiverso...

🌌 [Universo 1/2] Generando BTC-USD_Synthetic_U1...
   🔎 Corriendo Evaluador Walk-Forward (Expanding Window)...
Iniciando Walk-Forward (Ventana:360, Salto:20, Horizonte:150 velas en 15 bloques, Contexto:0)


Física Débil/Inexistente (R2=0.02). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.01). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=-0.00). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.00). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.04). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.04). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.00). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.00). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.00). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.00). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.00). Evitando iter

   ✅ Éxito. Generadas 40 iteraciones Walk-Forward.

🌌 [Universo 2/2] Generando BTC-USD_Synthetic_U2...
   🔎 Corriendo Evaluador Walk-Forward (Expanding Window)...
Iniciando Walk-Forward (Ventana:360, Salto:20, Horizonte:150 velas en 15 bloques, Contexto:0)


Física Débil/Inexistente (R2=0.02). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.00). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.04). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.01). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.01). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.01). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.01). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.01). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.01). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.01). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.01). Evitando itera

   ✅ Éxito. Generadas 40 iteraciones Walk-Forward.

🎉 Experimento Multiverso completado en 388.40 segundos!


## 💾 4. Exportación e Integración de Resultados
Concatenamos los resultados Walk-Forward de todos los universos paralelos. Guardamos el archivo `macro_backtest_synthetic_db.csv` y, por conveniencia, hacemos un respaldo del real `macro_backtest_predictive_db.csv` y lo sobrescribimos con los datos sintéticos para permitir una auditoría directa e instantánea en el Notebook 3.

In [25]:
if len(all_results) > 0:
    df_final = pd.concat(all_results, ignore_index=True)
    
    # 1. Guardar DB Sintética limpia
    df_final.to_csv(DB_SYNTHETIC_PATH, index=False)
    print(f"💾 DB Sintética guardada en: {os.path.abspath(DB_SYNTHETIC_PATH)}")
    
    # 2. Respaldo y Sobrescritura del CSV Predictivo Real para análisis instantáneo
    if os.path.exists(DB_PREDICTIVE_PATH):
        backup_path = DB_PREDICTIVE_PATH + ".bak"
        if not os.path.exists(backup_path):
            pd.read_csv(DB_PREDICTIVE_PATH).to_csv(backup_path, index=False)
            print(f"🗄️ Creado respaldo de base de datos predictiva real en: {backup_path}")
            

    
    # 3. Guardar caché física de universos para la calibración del Notebook 3
    with open(CACHE_PATH, "wb") as f:
        pickle.dump(synthetic_universes_cache, f)
    print(f"🔑 Caché física de multiversos guardada en: {os.path.abspath(CACHE_PATH)}")
else:
    print("❌ ERROR: Ningún resultado fue recopilado de la simulación masiva.")

💾 DB Sintética guardada en: c:\Users\ussaa\Documents\KineTopus Engine\notebooks_val\macro_backtest_synthetic_db.csv
🔑 Caché física de multiversos guardada en: c:\Users\ussaa\Documents\KineTopus Engine\notebooks_val\.synthetic_prices.pkl


## 📊 5. Análisis Rápido In-Situ (Tasa de Mortalidad y Edge)
Ejecutamos un diagnóstico inicial para evaluar de forma directa la tasa de fallos matemáticos y la precisión direccional (Hit Ratio) promedio en el Tramo 1 frente al azar.

In [26]:
if len(all_results) > 0:
    # A. Tasa de Mortalidad
    mortalidad = df_final.groupby(['Validez']).size().reset_index(name='Iteraciones')
    mortalidad['Porcentaje (%)'] = round((mortalidad['Iteraciones'] / len(df_final)) * 100, 2)
    print("\n" + "="*80)
    print(" 📊 INFORME DE MORTALIDAD FÍSICA EN EL MULTIVERSO:")
    print("="*80)
    print(mortalidad.to_string(index=False))
    print("="*80)
    
    # B. Hit Ratio Promedio del Tramo 1 (Bloque 1) vs Azar
    df_ok = df_final[df_final['Validez'] == 'OK'].copy()
    if not df_ok.empty:
        hit_b1_promedio = df_ok['Hit_B1'].mean() * 100
        print(f"\n🎯 Hit Ratio Direccional Promedio del Tramo 1 (Bloque 1): {hit_b1_promedio:.2f}% (Azar = 50%)")
        
        # Calcular decaimiento direccional por tramos
        hit_cols = [f'Hit_B{i}' for i in range(1, 16)]
        decay_curve = df_ok[hit_cols].mean() * 100
        
        # Gráfico interactivo de la Curva de Decaimiento
        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=list(range(1, 16)), y=decay_curve.values,
            mode='lines+markers',
            name='Hit Ratio SINDy',
            line=dict(color='#00ffcc', width=3),
            marker=dict(size=8, symbol='circle')
        ))
        fig.add_hline(y=50, line_dash="dash", line_color="red", annotation_text="Zona de Ceguera (Ruido - 50%)")
        fig.update_layout(
            title="⏳ Vida Útil de la Ecuación Física (Decaimiento en Universos Sintéticos)",
            xaxis_title="Tramo de Predicción (Bloque de 10 velas)",
            yaxis_title="Precisión Direccional (%)",
            yaxis_range=[40, 75],
            template="plotly_dark",
            height=500
        )
        fig.show()
    else:
        print("⚠️ No existen iteraciones con Validez = OK para calcular el Hit Ratio.")
else:
    print("Sin datos para analizar.")


 📊 INFORME DE MORTALIDAD FÍSICA EN EL MULTIVERSO:
         Validez  Iteraciones  Porcentaje (%)
FALLO MATEMÁTICO           19           23.75
              OK           61           76.25

🎯 Hit Ratio Direccional Promedio del Tramo 1 (Bloque 1): 40.98% (Azar = 50%)
